# How Much Do Seattlites Pay For Housing
This notebook produces an interactive choropleth web map that showcases the median housing cost as a percentage of monthly income for all cencus tracts within Seattle.

## Data Sources
2020 Seattle Census Tracts: https://data.seattle.gov/dataset/2020-Census-Tracts-Seattle/i8nh-jyuw/about_data  
5-year American Community Survey: https://www.census.gov/data/developers/data-sets/acs-5year.html

In [1]:
# imports
import geopandas as gpd
import folium
import matplotlib
import os
from dotenv import load_dotenv
import requests
import pandas as pd

In [2]:
# load environment
load_dotenv()
CARTO_API_KEY = os.getenv('CARTO_BASEMAPS_API_KEY')
CENSUS_API_KEY = os.getenv('US_CENSUS_API_KEY')

## Importing data, geometry, and tileset
Table IDs for the relevant variables were gathered from the ACS 5-year Survey technical documentation. Population, Income, and Housing data are pulled via API call and merged with a geodataframe containing the census tracts within Seattle. The basemap tileset is pulled from Carto via API.

In [3]:
# Table IDs for Total Population, Median Household Income, and Median Housing Cost
variables = "B01003_001E,B19013_001E,B25105_001E"

url = (
    f"https://api.census.gov/data/2024/acs/acs5?get={variables}&for=tract:*&in=state:53+county:033&key={CENSUS_API_KEY}"
)

response = requests.get(url)
data = response.json()

tracts_gdf = pd.DataFrame(data[1:], columns=data[0])
tracts_gdf = tracts_gdf.rename(columns={"B01003_001E": "population", "B19013_001E":"median_income", "B25105_001E":"median_monthly_housing_cost",})
tracts_gdf["GEOID"] = tracts_gdf["state"] + tracts_gdf["county"] + tracts_gdf["tract"]

# Load the Seattle Census tracts .geojson file and merge it with ACS data.
seattle_tracts_gdf = gpd.read_file("2020_Census_Tracts_Seattle.geojson")
seattle_tracts_gdf = seattle_tracts_gdf.merge(tracts_gdf, left_on="GEOID20", right_on="GEOID", how="inner")
seattle_tracts_gdf["median_income"] = seattle_tracts_gdf["median_income"].astype(int)
seattle_tracts_gdf["population"] = seattle_tracts_gdf["population"].astype(int)
seattle_tracts_gdf["median_monthly_income"] = (seattle_tracts_gdf["median_income"] / 12).astype(int)
seattle_tracts_gdf["median_monthly_housing_cost"] = seattle_tracts_gdf["median_monthly_housing_cost"].astype(int)
seattle_tracts_gdf.drop(labels=["state", "county", "tract"], axis=1, inplace=True)

print(seattle_tracts_gdf.head())
print(seattle_tracts_gdf.dtypes)


# Save Carto basemap tileset to variable
tiles = f"https://basemaps.cartocdn.com/rastertiles/voyager/{{z}}/{{x}}/{{y}}.png?key={CARTO_API_KEY}"

   OBJECTID      GEOID20  GROSS_ACRES  LAND_ACRES  WATER_ACRES  \
0         1  53033000402   444.493643  444.527261     0.000000   
1         2  53033000403   180.930716  165.408389    15.535762   
2         3  53033000700   319.838996  319.862596     0.000000   
3         4  53033003302   154.942826  154.952619     0.000000   
4         5  53033003601   166.571397  166.583375     0.000000   

                 NAME  TRACT_NUMB BASENAME               UVDA_AREA  CRA_NO  \
0   Census Tract 4.02         402     4.02     Bitter Lake Village     9.1   
1   Census Tract 4.03         403     4.03     Bitter Lake Village     9.1   
2      Census Tract 7         700        7               Lake City     8.2   
3  Census Tract 33.02        3302    33.02                 Ballard    10.2   
4  Census Tract 36.01        3601    36.01  Green Lake - Roosevelt     9.4   

   CRA_GRP                      GEN_ALIAS  \
0        9          Broadview/Bitter Lake   
1        9          Broadview/Bitter Lake   

## Creating display fields, transformations, and map object
Display fields are derived from their respective income, housing costs, and population fields via string transformations. The "housing_expense_percentage" field is derived from housing cost and income fields. The ensuing geodataframe is used to create the interactive choropleth web map. The quantile display scheme shows that households in most Seattle census tracts (on average) spend 19-26% of their monthly income on housing.

Significant outliers include the census tracts within the University District, demographically unique in that it includes a large population of young adults with no income of their own. This may explain why census tract 53.04, of which a significant portion is made up of University of Washington's West Campus dormitories, has an average monthly housing cost percentage calculated at 119.48% of household income.

In [7]:
# create fields meant for display on the web map and tables
seattle_tracts_gdf["median_income_display"] = seattle_tracts_gdf["median_income"].apply(lambda x: f"${x:,.0f}")
seattle_tracts_gdf["median_monthly_income_display"] = seattle_tracts_gdf["median_monthly_income"].apply(lambda x: f"${x:,.0f}")
seattle_tracts_gdf["median_monthly_housing_cost_display"] = seattle_tracts_gdf["median_monthly_housing_cost"].apply(lambda x: f"${x:,.0f}")
seattle_tracts_gdf["population_display"] = seattle_tracts_gdf["population"].apply(lambda x: f"{x:,.0f}")

# calculate housing cost as a percentage of income per census tract
seattle_tracts_gdf["housing_expense_percentage"] = (seattle_tracts_gdf["median_monthly_housing_cost"] / seattle_tracts_gdf["median_monthly_income"]) * 100
seattle_tracts_gdf["housing_expense_percentage"] = seattle_tracts_gdf["housing_expense_percentage"].astype(float)
seattle_tracts_gdf["housing_expense_percentage_display"] = seattle_tracts_gdf["housing_expense_percentage"].round(2).astype(str).add(' %')

# create web map
m = seattle_tracts_gdf.explore(
    tiles=tiles,
    column="housing_expense_percentage",
    tooltip=["GEN_ALIAS","BASENAME","population_display","median_monthly_housing_cost_display", "median_income_display","housing_expense_percentage_display"],
    tooltip_kwds={
        "aliases":["Neighborhood:", "Census Tract:", "Population:", "Median Monthly Housing Cost:", "Median Annual Household Income:", "Median Housing Cost Percentage:"]
    },
    scheme="Quantiles",
    cmap="Purples",
    legend=True,
    legend_kwds={
        "caption": "Median Housing Cost as a Percentage of Household Income",
        "style": "color: white!important;"
    },
    min_zoom=11,
    max_bounds=True,
    min_lat=47.40,
    max_lat=47.78,
    min_lon=-122.55,
    max_lon=-122.15,
    attr='© <a href="https://carto.com/attributions">CARTO</a> | Data from the <a href="https://www.census.gov/programs-surveys/acs">American Community Survey</a> and <a href="https://data.seattle.gov/">Seattle Open Data Portal</a>'
)
m

## Output to static HTML dashboard
Also included are a pair of dataframes deriving the census tracts with the highest average housing costs and incomes, for additional context.

In [8]:
# create context tables
top_housing_cost = seattle_tracts_gdf.nlargest(10, "median_monthly_housing_cost")[["GEN_ALIAS", "median_monthly_housing_cost_display"]]
top_income = seattle_tracts_gdf.nlargest(10, "median_income")[["GEN_ALIAS", "median_income_display"]]

# convert to HDMI
def df_to_html_table(df, columns_rename):
    styled = df.rename(columns=columns_rename)
    return styled.to_html(index=False, classes="dashboard-table", border=0)

table1_html = df_to_html_table(top_housing_cost, {"GEN_ALIAS":"Neighborhood", "median_monthly_housing_cost_display": "Median Monthly Housing Cost"})
table2_html = df_to_html_table(top_income, {"GEN_ALIAS":"Neighborhood", "median_income_display":"Median Annual Income"})

In [9]:
# output map to .html file
m.save("map.html")

# combine map and tables into HTML block, and write to file
dashboard_html = f"""
<!DOCTYPE html>
<html>
<head>
<style>
    body {{font-family: sans-serif; margin: 20px; background: #f5f5f5;}}
    .dashboard-grid {{
        display: grid;
        grid-template-columns: 2fr 1fr;
        gap: 20px;
    }}
    .map-container iframe {{width: 100%; height: 600px;}}
    .tables-container {{display: flex; flex-direction: column;}}
    .dashboard-table {{width: 100%; background: white; border-radius: 6px; }}
    .dashboard-table th {{background: #333; color: white; padding: 8px; text-align: left;}}
    .dashboard-table td {{ padding: 6px 8px; border-bottom: 1px solid #eee; }}
    h3 {{ margin-bottom: 6px; }}
</style>
</head>
<body>
    <h1>How Much Do Seattlites Pay For Housing?</h1>
    
    <div class="dashboard-grid">
        <div class="map-container">
            <iframe src="map.html"></iframe>
        </div>
        <div class="tables-container">
            <div><h3>Highest Housing Costs</h3>{table1_html}</div>
            <div><h3>Highest Median Income</h3>{table2_html}</div>
        </div>
    </div>
</body>
</html>
"""

with open("dashboard.html", "w") as f:
    f.write(dashboard_html)